# Chunking Strategies & Hybrid Search

**Goal**: Upgrade from BM25-only to hybrid search that understands both *what users type* and *what they mean*.

By the end of this notebook you will understand:
- Why naive 512-token chunking destroys retrieval quality for academic papers
- The section-aware chunking algorithm and its 4 cases
- What embedding dimensions mean and why 1024 is a good choice
- How Jina AI v3 embeddings work and how to call them efficiently
- How `knn_vector` fields work in OpenSearch (HNSW algorithm)
- Reciprocal Rank Fusion — the math, the k constant, why rank beats raw score
- OpenSearch's native `hybrid` query combining BM25 + kNN
- Graceful degradation: hybrid → BM25 when the embedding service is down

---

## The Architecture Upgrade

```
Before (BM25 only):     query → multi_match → ranked docs

After (Hybrid):         query ──► BM25 (multi_match) ──► rank list A ──┐
                              ├──► Embedding (Jina v3) ──► kNN search ──► RRF fusion ──► final ranking
                              └──► [fallback: BM25 only if embedding fails]
```

**What changes in the data model:**
- `papers` index: unchanged (BM25 still works as before)
- NEW `chunks` index: one document per *chunk* with a `knn_vector` embedding field

We search chunks, not full papers. This is the core architectural shift.

---

## Section 1: Why Chunking Matters — And Why Naive Chunking Fails

### The problem with 512-token fixed chunks

Most tutorials say: "just split every 512 tokens with 50-token overlap". Here's why that's wrong for academic papers:

```
Paper structure:
  Abstract (300 words)
  Introduction (800 words)         ← semantically coherent unit
  Related Work (1200 words)        ← semantically coherent unit
  Methods (2000 words)             ← semantically coherent unit
  Results (1500 words)             ← semantically coherent unit
  Conclusion (400 words)

Fixed 512-token chunking:
  Chunk 1: Abstract + half of Introduction    ← two different ideas mixed
  Chunk 2: Rest of Introduction + half of Related Work  ← boundary breaks context
  Chunk 3: Rest of Related Work + start of Methods     ← same problem
  ...
```

Fixed chunks **ignore document structure**. A chunk covering half Introduction + half Related Work embeds into a vector that represents neither concept well — it gets recalled for neither query.

### Section-aware chunking: the algorithm

We have 4 cases based on section length:

| Section size | Strategy | Why |
|---|---|---|
| < 100 words | Merge with next section | Too small to embed meaningfully |
| 100–800 words | Keep as one chunk | Perfect size for embeddings |
| > 800 words | Sliding window split | Preserve overlap for context continuity |
| No sections | Paragraph-based fallback | Works for unstructured PDFs |

In [2]:
import uuid
import re
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class TextChunk:
    """A single chunk ready for embedding and indexing."""
    chunk_id:     str
    arxiv_id:     str
    chunk_text:   str    # the content that gets embedded
    section_name: str    # which section this came from
    chunk_index:  int    # 0-based position within the paper
    word_count:   int


class TextChunker:
    """
    Section-aware chunker for academic papers.

    Each chunk includes a context header:
        Title: {paper title}
        Abstract: {abstract snippet}
        Section: {section_name}
        {section content}

    This header gives the embedding model context about the paper
    even when the chunk content is dense/technical. It's the single
    most impactful quality improvement over naive chunking.
    """

    def __init__(
        self,
        chunk_size: int = 600,    # target words per chunk
        overlap_size: int = 100,  # overlap words between sliding-window chunks
        min_chunk_size: int = 100, # below this, merge with next section
    ):
        self.chunk_size    = chunk_size
        self.overlap_size  = overlap_size
        self.min_chunk_size = min_chunk_size

    def chunk_paper(
        self,
        arxiv_id: str,
        title: str,
        abstract: str,
        full_text: str,
        sections: Optional[dict[str, str]] = None,
    ) -> list[TextChunk]:
        """
        Chunk a paper into semantically coherent pieces.

        Tries section-aware chunking first (when Docling extracted sections).
        Falls back to paragraph-based chunking for unstructured text.
        """
        # Build context header — prepended to every chunk
        abstract_snippet = " ".join(abstract.split()[:80])  # first ~80 words
        header = f"Title: {title}\nAbstract: {abstract_snippet}\n\n"

        if sections and len(sections) >= 2:
            return self._chunk_by_sections(arxiv_id, header, sections)
        else:
            return self._chunk_by_paragraphs(arxiv_id, header, full_text)

    def _chunk_by_sections(self, arxiv_id: str, header: str, sections: dict) -> list[TextChunk]:
        """Case 1/2/3: section-aware chunking with merge/keep/split logic."""
        chunks = []
        chunk_index = 0
        sections_list = list(sections.items())
        i = 0

        while i < len(sections_list):
            section_name, content = sections_list[i]
            words = content.split()
            wc = len(words)

            # Case 1: Too small — merge with next section
            if wc < self.min_chunk_size and i + 1 < len(sections_list):
                next_name, next_content = sections_list[i + 1]
                merged_content = content + "\n\n" + next_content
                merged_name    = f"{section_name} + {next_name}"
                sections_list[i + 1] = (merged_name, merged_content)
                i += 1
                continue

            # Case 2: Perfect size — use as single chunk
            if wc <= self.chunk_size:
                chunk_text = f"{header}Section: {section_name}\n\n{content}"
                chunks.append(TextChunk(
                    chunk_id=str(uuid.uuid4()),
                    arxiv_id=arxiv_id,
                    chunk_text=chunk_text,
                    section_name=section_name,
                    chunk_index=chunk_index,
                    word_count=len(chunk_text.split()),
                ))
                chunk_index += 1

            # Case 3: Too large — sliding window split
            else:
                sub_chunks = self._sliding_window(words, self.chunk_size, self.overlap_size)
                for j, sub in enumerate(sub_chunks):
                    chunk_text = f"{header}Section: {section_name} (part {j+1}/{len(sub_chunks)})\n\n{sub}"
                    chunks.append(TextChunk(
                        chunk_id=str(uuid.uuid4()),
                        arxiv_id=arxiv_id,
                        chunk_text=chunk_text,
                        section_name=f"{section_name} (part {j+1})",
                        chunk_index=chunk_index,
                        word_count=len(chunk_text.split()),
                    ))
                    chunk_index += 1

            i += 1

        return chunks

    def _chunk_by_paragraphs(self, arxiv_id: str, header: str, full_text: str) -> list[TextChunk]:
        """Case 4: fallback — split on paragraph breaks, then sliding window."""
        # Split on double newlines (paragraph boundaries)
        paragraphs = [p.strip() for p in re.split(r'\n\n+', full_text) if p.strip()]
        # Combine short paragraphs greedily until we hit chunk_size
        combined = []
        current_words = []
        for para in paragraphs:
            para_words = para.split()
            if current_words and len(current_words) + len(para_words) > self.chunk_size:
                combined.append(" ".join(current_words))
                current_words = para_words[-self.overlap_size:]  # seed next chunk with overlap
            current_words.extend(para_words)
        if current_words:
            combined.append(" ".join(current_words))

        chunks = []
        for i, text in enumerate(combined):
            chunk_text = f"{header}Section: Content\n\n{text}"
            chunks.append(TextChunk(
                chunk_id=str(uuid.uuid4()),
                arxiv_id=arxiv_id,
                chunk_text=chunk_text,
                section_name="Content",
                chunk_index=i,
                word_count=len(chunk_text.split()),
            ))
        return chunks

    def _sliding_window(self, words: list, size: int, overlap: int) -> list[str]:
        """Split a word list into overlapping windows of `size` words."""
        step   = size - overlap
        result = []
        start  = 0
        while start < len(words):
            window = words[start:start + size]
            result.append(" ".join(window))
            if start + size >= len(words):
                break
            start += step
        return result


# --- Demo ---
chunker = TextChunker(chunk_size=600, overlap_size=100, min_chunk_size=100)

sample_sections = {
    "Abstract":      "We present a new approach to attention...",  # 7 words → merge
    "Introduction":  " ".join(["word"] * 350),    # 350 words → keep as-is
    "Related Work":  " ".join(["word"] * 280),    # 280 words → keep as-is
    "Methods":       " ".join(["word"] * 1200),   # 1200 words → split into 2
    "Results":       " ".join(["word"] * 450),    # 450 words → keep
    "Conclusion":    " ".join(["word"] * 200),    # 200 words → keep
}

chunks = chunker.chunk_paper(
    arxiv_id="2301.00001",
    title="Attention Is All You Need",
    abstract="We propose a novel architecture based on attention mechanisms...",
    full_text="",
    sections=sample_sections,
)

print(f"Paper chunked into {len(chunks)} chunks:")
for c in chunks:
    print(f"  [{c.chunk_index}] {c.section_name:<35} {c.word_count:>5} words")

print(f"\nChunk 0 text preview (first 300 chars):")
print(chunks[0].chunk_text[:300])

Paper chunked into 7 chunks:
  [0] Abstract + Introduction               377 words
  [1] Related Work                          299 words
  [2] Methods (part 1)                      620 words
  [3] Methods (part 2)                      620 words
  [4] Methods (part 3)                      220 words
  [5] Results                               468 words
  [6] Conclusion                            218 words

Chunk 0 text preview (first 300 chars):
Title: Attention Is All You Need
Abstract: We propose a novel architecture based on attention mechanisms...

Section: Abstract + Introduction

We present a new approach to attention...

word word word word word word word word word word word word word word word word word word word word word word word


### 💡 Key insight: The context header

Every chunk starts with:
```
Title: Attention Is All You Need
Abstract: We propose a novel architecture based on attention...

Section: Methods

{section content}
```

Without this header, a chunk containing dense equations from the Methods section embeds into a generic vector with no paper context. With the header, the embedding model knows this is about transformer attention — making the vector meaningful for retrieval.

This simple trick is responsible for a significant share of the quality improvement over naive chunking.

### 💡 Key insight: overlap_size in sliding window

When Methods is 1200 words and we split at 600:
```
Chunk A: words[0..599]    (600 words)
Chunk B: words[500..1099]  (600 words, starts 100 words before end of A)
Chunk C: words[1000..1199] (200 words — last fragment)
```
The 100-word overlap means a concept that spans the boundary is fully captured in at least one chunk.

---

## Section 2: Embeddings — From Text to Vectors

### What is an embedding?

An embedding model converts text into a dense vector of floating-point numbers. Semantically similar texts produce vectors that are close together in high-dimensional space.

```
"transformer attention mechanism"  →  [0.024, -0.183, 0.441, ...] (1024 numbers)
"multi-head self attention"        →  [0.031, -0.171, 0.438, ...]  ← close!
"chocolate chip cookies"           →  [-0.412, 0.892, -0.201, ...]  ← far away
```

### Why 1024 dimensions (Jina v3)?

| Dimensions | Quality | Storage/Speed | Model example |
|---|---|---|---|
| 384 | Good | Fast, small | all-MiniLM-L6-v2 |
| 768 | Better | Medium | BGE-base |
| 1024 | High | Moderate | Jina v3, BGE-large |
| 3072 | Highest | Slow, large | OpenAI text-embedding-3-large |

1024 is the sweet spot for academic retrieval: high quality without the storage/latency cost of 3072. For our OpenSearch index, each chunk vector = 1024 × 4 bytes = 4KB. 10,000 chunks = ~40MB of vector data — very manageable.

### Why Jina AI v3?
- Free API key with no signup required: https://jina.ai/embeddings/
- 1024 dimensions, optimised for retrieval tasks
- Handles academic content well (technical terminology)
- Supports `task` parameter: `retrieval.query` vs `retrieval.passage` (different embeddings for queries vs documents — asymmetric embedding)

In [3]:
import os
import time
import numpy as np
import requests

JINA_API_KEY = os.environ.get("JINA_API_KEY", "")
JINA_URL     = "https://api.jina.ai/v1/embeddings"
JINA_MODEL   = "jina-embeddings-v3"
EMBEDDING_DIM = 1024

def embed_texts(texts: list[str], task: str = "retrieval.passage") -> list[list[float]]:
    """
    Call Jina AI to get embeddings for a list of texts.

    task options:
      'retrieval.passage' — for document chunks (at index time)
      'retrieval.query'   — for user queries (at search time)

    Asymmetric embeddings: using the right task type improves retrieval
    because queries and passages have different linguistic patterns.
    """
    if not JINA_API_KEY:
        print("⚠️  No JINA_API_KEY set — returning zero vectors for demo")
        return [[0.0] * EMBEDDING_DIM for _ in texts]

    headers = {
        "Authorization": f"Bearer {JINA_API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model":    JINA_MODEL,
        "task":     task,
        "input":    texts,
        "dimensions": EMBEDDING_DIM,
    }

    resp = requests.post(JINA_URL, headers=headers, json=payload, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    # Response: {"data": [{"embedding": [...], "index": 0}, ...]}
    return [item["embedding"] for item in sorted(data["data"], key=lambda x: x["index"])]


# Test: embed a few texts and check cosine similarity
test_texts = [
    "transformer self-attention mechanism for language models",
    "multi-head attention in neural networks",
    "reinforcement learning from human feedback",
]

print("Embedding 3 test texts...")
embeddings = embed_texts(test_texts, task="retrieval.passage")
print(f"✅ Got {len(embeddings)} embeddings, each {len(embeddings[0])} dimensions")


def cosine_similarity(a: list, b: list) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


print("\nCosine similarities:")
print(f"  Text 0 vs Text 1 (both about attention): {cosine_similarity(embeddings[0], embeddings[1]):.4f}")
print(f"  Text 0 vs Text 2 (different topics):     {cosine_similarity(embeddings[0], embeddings[2]):.4f}")
print("  (Higher = more similar. Expect 0 vs 1 >> 0 vs 2)")

Embedding 3 test texts...
✅ Got 3 embeddings, each 1024 dimensions

Cosine similarities:
  Text 0 vs Text 1 (both about attention): 0.5309
  Text 0 vs Text 2 (different topics):     0.4205
  (Higher = more similar. Expect 0 vs 1 >> 0 vs 2)


In [4]:
# Batch embedding — critical for indexing efficiency
# Jina can handle up to 2048 texts per request
# We batch in groups of 32 to stay safe and respect rate limits

def embed_chunks_batched(chunk_texts: list[str], batch_size: int = 32) -> list[list[float]]:
    """
    Embed a large list of texts in batches.

    Why batch?
    - One API call per chunk is ~200ms × 10,000 chunks = 33 minutes
    - Batching 32 texts per call: ~200ms × 312 calls = 1 minute
    """
    all_embeddings = []
    for i in range(0, len(chunk_texts), batch_size):
        batch = chunk_texts[i:i + batch_size]
        batch_embeddings = embed_texts(batch, task="retrieval.passage")
        all_embeddings.extend(batch_embeddings)
        if i + batch_size < len(chunk_texts):
            time.sleep(0.1)  # gentle rate limiting
    return all_embeddings


# Embed all chunks from our paper demo
chunk_texts = [c.chunk_text for c in chunks]
print(f"Embedding {len(chunk_texts)} chunks...")
chunk_embeddings = embed_chunks_batched(chunk_texts)
print(f"✅ Got {len(chunk_embeddings)} embeddings")

# Pair each chunk with its embedding
for chunk, embedding in zip(chunks, chunk_embeddings):
    print(f"  [{chunk.chunk_index}] {chunk.section_name:<35} embedding[0:3]={embedding[:3]}")

Embedding 7 chunks...
✅ Got 7 embeddings
  [0] Abstract + Introduction             embedding[0:3]=[-0.00270121, -0.01565675, 0.15484859]
  [1] Related Work                        embedding[0:3]=[-0.00387531, -0.00254842, 0.14658071]
  [2] Methods (part 1)                    embedding[0:3]=[0.0117836, -0.035262, 0.15666391]
  [3] Methods (part 2)                    embedding[0:3]=[0.01164189, -0.03401147, 0.15681404]
  [4] Methods (part 3)                    embedding[0:3]=[0.00339742, 0.00059601, 0.14712508]
  [5] Results                             embedding[0:3]=[0.00781204, -0.02221912, 0.15152164]
  [6] Conclusion                          embedding[0:3]=[0.0115451, 0.00076797, 0.1448018]


### 💡 Key insight: `retrieval.query` vs `retrieval.passage`

Jina v3 (and models like E5) use **asymmetric embeddings**: queries and passages are embedded differently. The model is trained so that query embeddings align well with passage embeddings, even though they're different vector representations.

Always use:
- `retrieval.passage` when embedding chunks at index time
- `retrieval.query` when embedding the user's search query at search time

Using the wrong task type can silently degrade retrieval quality.

---

## Section 3: OpenSearch — Adding the Vector Field

### Concept: A separate `chunks` index

We don't add the vector to the `papers` index. We create a new `chunks` index where each document is one chunk.

Why separate?
- A paper with 18 chunks would need 18 nested knn vectors — complex to query
- Separate index: one document = one chunk = one vector = simple flat kNN search
- The `papers` index (BM25) stays unchanged — no breaking change
- Each chunk document carries its `arxiv_id` back-reference to join with paper metadata

In [5]:
# The chunks index config — the critical new field is 'embedding'

CHUNKS_INDEX_CONFIG = {
    "settings": {
        "number_of_shards":   1,
        "number_of_replicas": 0,
        "index.knn": True,    # MUST be True to enable kNN search
    },
    "mappings": {
        "properties": {
            # Paper back-reference
            "arxiv_id":     {"type": "keyword"},
            # Chunk identity
            "chunk_id":     {"type": "keyword"},
            "chunk_index":  {"type": "integer"},
            "section_name": {"type": "keyword"},
            "word_count":   {"type": "integer"},
            # The chunk text — BM25 searchable
            "chunk_text":   {"type": "text", "analyzer": "standard"},
            # Paper metadata copied here for context in search results
            "title":        {"type": "text"},
            "abstract":     {"type": "text"},
            "categories":   {"type": "keyword"},
            "published_at": {"type": "date"},
            # ── The vector field ───────────────────────────────────────
            # knn_vector: stores a dense float vector
            # dimension: MUST match exactly what the embedding model outputs
            # method: HNSW (Hierarchical Navigable Small World)
            #   - approximate nearest-neighbour algorithm
            #   - much faster than brute-force at scale
            #   - space_type: cosinesimil (cosine similarity distance)
            "embedding": {
                "type": "knn_vector",
                "dimension": 1024,
                "method": {
                    "name":       "hnsw",
                    "space_type": "cosinesimil",
                    "engine":     "lucene",
                    "parameters": {
                        "m":            16,   # HNSW connections per node
                        "ef_construction": 128, # build-time quality/speed tradeoff
                    }
                }
            }
        }
    }
}

print("Chunks index config — key field: 'embedding'")
import json
print(json.dumps(CHUNKS_INDEX_CONFIG["mappings"]["properties"]["embedding"], indent=2))

Chunks index config — key field: 'embedding'
{
  "type": "knn_vector",
  "dimension": 1024,
  "method": {
    "name": "hnsw",
    "space_type": "cosinesimil",
    "engine": "lucene",
    "parameters": {
      "m": 16,
      "ef_construction": 128
    }
  }
}


### 💡 Key insight: HNSW — approximate nearest-neighbour

With 10,000 chunks × 1024 dimensions, **brute-force** cosine similarity requires computing 10,000 dot products per query — fine for small data, but slow at scale.

**HNSW** builds a graph structure at index time where each vector is connected to its nearest neighbours. At query time, it navigates this graph rather than checking every vector.

Tradeoffs:
- `m=16`: each node has 16 graph connections. Higher = better recall, more memory
- `ef_construction=128`: how thoroughly to search during index building. Higher = better graph, slower build

For our dev setup (a few thousand chunks), the defaults are fine. For millions of chunks in production, you'd tune these.

In [30]:
from opensearchpy import OpenSearch

client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_compress=True, use_ssl=False, verify_certs=False,
)

CHUNKS_INDEX = "chunks"

# Create the chunks index
try:
    if client.indices.exists(index=CHUNKS_INDEX):
        print(f"Index '{CHUNKS_INDEX}' already exists")
    else:
        client.indices.create(index=CHUNKS_INDEX, body=CHUNKS_INDEX_CONFIG)
        print(f"✅ Index '{CHUNKS_INDEX}' created")
except Exception as e:
    print(f"❌ {e}")

Index 'chunks' already exists


In [11]:
# Index our chunks with embeddings
from opensearchpy.helpers import bulk

def index_chunks_with_embeddings(
    chunks: list,
    embeddings: list,
    paper_meta: dict,   # title, abstract, categories, published_at
) -> dict:
    """
    Bulk-index chunks paired with their embeddings.

    chunk_id is used as the OpenSearch _id:
    - Idempotent: re-indexing a paper replaces its chunks
    - Since chunk_id is a UUID, same paper re-chunked = new UUIDs = clean replace
    """
    actions = []
    for chunk, embedding in zip(chunks, embeddings):
        doc = {
            "arxiv_id":    chunk.arxiv_id,
            "chunk_id":    chunk.chunk_id,
            "chunk_index": chunk.chunk_index,
            "section_name": chunk.section_name,
            "word_count":  chunk.word_count,
            "chunk_text":  chunk.chunk_text,
            "embedding":   embedding,          # the 1024-float vector
            # Paper metadata for context in results
            "title":        paper_meta.get("title", ""),
            "abstract":     paper_meta.get("abstract", ""),
            "categories":   paper_meta.get("categories", []),
            "published_at": paper_meta.get("published_at"),
        }
        actions.append({
            "_op_type": "index",
            "_index":   CHUNKS_INDEX,
            "_id":      chunk.chunk_id,
            "_source":  doc,
        })

    success, errors = bulk(client, actions, raise_on_error=False, stats_only=False)
    return {"indexed": success, "errors": len(errors) if isinstance(errors, list) else errors}


# Index our sample chunks
paper_meta = {
    "title":       "Attention Is All You Need",
    "abstract":    "We propose a novel architecture based on attention mechanisms...",
    "categories":  ["cs.LG", "cs.CL"],
    "published_at": "2017-06-12",
}

result = index_chunks_with_embeddings(chunks, chunk_embeddings, paper_meta)
client.indices.refresh(index=CHUNKS_INDEX)  # immediate visibility (dev only)
print(f"Indexed: {result}")

total = client.count(index=CHUNKS_INDEX)["count"]
print(f"Total chunks in index: {total}")

Indexed: {'indexed': 7, 'errors': 0}
Total chunks in index: 7


---

## Section 4: Reciprocal Rank Fusion — The Math

### Why not just average scores?

BM25 scores and cosine similarity scores are on completely different scales:
- BM25: typically 0.5–5.0 (depends on corpus size, TF, IDF)
- Cosine: always 0.0–1.0

Simply averaging them would let one system dominate. **RRF uses ranks instead of scores** — it doesn't matter what the raw score is, only where each system ranked the document.

### The formula

$$\text{RRF}(d) = \sum_{s \in \{\text{BM25, kNN}\}} \frac{1}{k + \text{rank}_s(d)}$$

Where:
- $k = 60$ (constant — empirically found to work well)
- $\text{rank}_s(d)$ = position of document $d$ in result list from system $s$ (1-indexed)

A document ranked **1st** in both systems gets: $\frac{1}{60+1} + \frac{1}{60+1} = 0.0328$

A document ranked **100th** in one system gets almost no contribution from that system: $\frac{1}{160} = 0.00625$

In [12]:
# Demonstrate RRF with a concrete example

def rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank)

def compute_rrf(bm25_ranks: dict, knn_ranks: dict, k: int = 60) -> dict:
    """
    Fuse two ranked lists using RRF.

    bm25_ranks: {doc_id: rank}  (rank=1 is best)
    knn_ranks:  {doc_id: rank}
    Returns: {doc_id: rrf_score} sorted by score desc
    """
    all_docs = set(bm25_ranks) | set(knn_ranks)
    # Default rank = inf if doc not in one of the result sets
    # (contributes essentially 0 to RRF score)
    MAX_RANK = 1000

    scores = {}
    for doc in all_docs:
        r_bm25 = bm25_ranks.get(doc, MAX_RANK)
        r_knn  = knn_ranks.get(doc,  MAX_RANK)
        scores[doc] = rrf_score(r_bm25, k) + rrf_score(r_knn, k)

    return dict(sorted(scores.items(), key=lambda x: -x[1]))


# Concrete example: 5 papers, two retrieval systems
bm25_results = {"Paper-A": 1, "Paper-B": 2, "Paper-C": 3, "Paper-D": 4, "Paper-E": 5}
knn_results  = {"Paper-C": 1, "Paper-A": 2, "Paper-F": 3, "Paper-B": 4, "Paper-G": 5}
# Paper-F and Paper-G only appear in kNN; Paper-D and Paper-E only in BM25

fused = compute_rrf(bm25_results, knn_results)

print(f"{'Doc':<12} {'BM25 rank':>10} {'kNN rank':>10} {'RRF score':>12}")
print("-" * 46)
for doc, score in fused.items():
    bm25_r = bm25_results.get(doc, "—")
    knn_r  = knn_results.get(doc, "—")
    print(f"{doc:<12} {str(bm25_r):>10} {str(knn_r):>10} {score:>12.6f}")

print()
print("Observations:")
print("  - Paper-A (rank 1 BM25, rank 2 kNN) → highest RRF: appears in both, top positions")
print("  - Paper-C (rank 3 BM25, rank 1 kNN) → second: strong kNN signal boosts it")
print("  - Paper-F (not in BM25, rank 3 kNN) → can still surface via semantic signal alone")
print("  - Paper-D (rank 4 BM25, not in kNN) → lower: only one signal, lower rank")

Doc           BM25 rank   kNN rank    RRF score
----------------------------------------------
Paper-A               1          2     0.032522
Paper-C               3          1     0.032266
Paper-B               2          4     0.031754
Paper-F               —          3     0.016816
Paper-D               4          —     0.016568
Paper-E               5          —     0.016328
Paper-G               —          5     0.016328

Observations:
  - Paper-A (rank 1 BM25, rank 2 kNN) → highest RRF: appears in both, top positions
  - Paper-C (rank 3 BM25, rank 1 kNN) → second: strong kNN signal boosts it
  - Paper-F (not in BM25, rank 3 kNN) → can still surface via semantic signal alone
  - Paper-D (rank 4 BM25, not in kNN) → lower: only one signal, lower rank


### 💡 Key insight: Why k=60?

The constant k prevents top-ranked documents from dominating. With k=60:
- Rank 1: 1/(60+1) = 0.0164
- Rank 10: 1/(60+10) = 0.0143  ← only 13% lower than rank 1
- Rank 100: 1/(60+100) = 0.0063 ← much less contribution

This makes RRF **robust to ranking noise** in either system. A document that's in the top 10 of both lists beats one that's rank 1 in only one. k=60 was found empirically across many retrieval benchmarks — it's now the standard default in OpenSearch's native hybrid query.

### OpenSearch native hybrid query

OpenSearch 2.10+ includes native RRF hybrid search. We use it directly:

In [13]:
def build_hybrid_query(
    query_text: str,
    query_embedding: list[float],
    categories: list[str] | None = None,
    size: int = 10,
    from_: int = 0,
) -> dict:
    """
    Build an OpenSearch hybrid query combining BM25 + kNN with RRF.

    The 'hybrid' query type is OpenSearch's native RRF implementation.
    It runs both sub-queries, ranks results by each, fuses with RRF.
    """
    # BM25 sub-query on the chunk_text field
    bm25_query = {
        "bool": {
            "must": [{
                "multi_match": {
                    "query":    query_text,
                    "fields":   ["chunk_text^2", "title^3", "abstract"],
                    "type":     "best_fields",
                    "operator": "or",
                }
            }],
            # Category filter applies to both sub-queries via the outer bool
            "filter": [{"terms": {"categories": categories}}] if categories else [],
        }
    }

    # kNN sub-query on the embedding field
    knn_query = {
        "knn": {
            "embedding": {
                "vector": query_embedding,
                "k": size * 2,    # over-fetch, let RRF trim to final size
            }
        }
    }

    # Hybrid query: OpenSearch runs both, applies RRF
    return {
        "from": from_,
        "size": size,
        "query": {
            "hybrid": {
                "queries": [bm25_query, knn_query]
            }
        },
        # search_pipeline is configured separately (see Section 5)
        "_source": {"excludes": ["embedding"]},  # don't return the huge vector
    }


def build_vector_only_query(query_embedding: list[float], size: int = 10) -> dict:
    """Pure kNN vector search — useful for debugging/comparison."""
    return {
        "size": size,
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": size,
                }
            }
        },
        "_source": {"excludes": ["embedding"]},
    }


print("Hybrid query structure:")
sample_embedding = [0.01] * 1024  # placeholder
q = build_hybrid_query("transformer attention", sample_embedding, categories=["cs.AI"])
print(json.dumps(q["query"], indent=2))

Hybrid query structure:
{
  "hybrid": {
    "queries": [
      {
        "bool": {
          "must": [
            {
              "multi_match": {
                "query": "transformer attention",
                "fields": [
                  "chunk_text^2",
                  "title^3",
                  "abstract"
                ],
                "type": "best_fields",
                "operator": "or"
              }
            }
          ],
          "filter": [
            {
              "terms": {
                "categories": [
                  "cs.AI"
                ]
              }
            }
          ]
        }
      },
      {
        "knn": {
          "embedding": {
            "vector": [
              0.01,
              0.01,
              0.01,
              0.01,
              0.01,
              0.01,
              0.01,
              0.01,
              0.01,
              0.01,
              0.01,
              0.01,
              0.01,
              0.

### Setting up the RRF search pipeline in OpenSearch

The `hybrid` query type requires a **search pipeline** with `normalization_processor` to enable RRF. This is a one-time setup, not per-query.

In [34]:
# Create the hybrid normalization pipeline — must be done once
PIPELINE_ID = "hybrid-rrf-pipeline"

# NOTE: This OpenSearch build does not support normalization technique='rrf'.
# Use min_max so the pipeline can be created successfully.
PIPELINE_CONFIG = {
    "description": "Hybrid normalization for BM25 + kNN search",
    "phase_results_processors": [
        {
            "normalization-processor": {
                "normalization": {"technique": "min_max"},
                "combination": {
                    "technique": "arithmetic_mean",
                    "parameters": {"weights": [0.5, 0.5]}  # equal weight to BM25 and kNN
                }
            }
        }
    ]
}

try:
    resp = client.transport.perform_request(
        method="PUT",
        url=f"/_search/pipeline/{PIPELINE_ID}",
        body=PIPELINE_CONFIG,
    )
    print(f"✅ Pipeline '{PIPELINE_ID}' created: {resp}")
except Exception as e:
    print(f"❌ Pipeline creation failed: {e}")
    print("Check that OpenSearch is reachable on localhost:9200 and supports normalization-processor.")

✅ Pipeline 'hybrid-rrf-pipeline' created: {'acknowledged': True}


---

## Section 5: Running Hybrid Search

Now let's run real searches and compare BM25, kNN, and hybrid.

In [35]:
import time

def search_bm25_chunks(query_text: str, size: int = 5) -> dict:
    """Pure BM25 search on the chunks index."""
    q = {
        "size": size,
        "query": {
            "multi_match": {
                "query":    query_text,
                "fields":   ["chunk_text^2", "title^3", "abstract"],
                "type":     "best_fields",
            }
        },
        "_source": {"excludes": ["embedding"]},
    }
    return client.search(index=CHUNKS_INDEX, body=q)


def search_vector_chunks(query_embedding: list[float], size: int = 5) -> dict:
    """Pure kNN vector search on the chunks index."""
    return client.search(index=CHUNKS_INDEX, body=build_vector_only_query(query_embedding, size))


def search_hybrid_chunks(query_text: str, query_embedding: list[float], size: int = 5) -> dict:
    """Hybrid BM25 + kNN search with RRF fusion."""
    q = build_hybrid_query(query_text, query_embedding, size=size)
    return client.search(
        index=CHUNKS_INDEX,
        body=q,
        params={"search_pipeline": PIPELINE_ID},
    )


def print_results(response: dict, label: str):
    hits  = response["hits"]["hits"]
    total = response["hits"]["total"]["value"]
    took  = response["took"]
    print(f"\n{'='*55}")
    print(f"  {label}  |  {total} total  |  {took}ms")
    print(f"{'='*55}")
    for h in hits:
        src = h["_source"]
        score = h.get('_score') or 0
        print(f"  [{score:.4f}] arxiv:{src.get('arxiv_id')} | section:{src.get('section_name')}")
        text_preview = (src.get('chunk_text') or '')[:120].replace('\n', ' ')
        print(f"           {text_preview}...")


# Run all three search modes
QUERY = "attention mechanism in transformer models"

# Embed the query with retrieval.query task (NOT passage — this matters!)
query_embedding = embed_texts([QUERY], task="retrieval.query")[0]

t1 = time.time(); bm25_resp    = search_bm25_chunks(QUERY);                     bm25_ms    = int((time.time()-t1)*1000)
t2 = time.time(); vector_resp  = search_vector_chunks(query_embedding);          vector_ms  = int((time.time()-t2)*1000)
t3 = time.time(); hybrid_resp  = search_hybrid_chunks(QUERY, query_embedding);  hybrid_ms  = int((time.time()-t3)*1000)

print_results(bm25_resp,   f"BM25 ({bm25_ms}ms)")
print_results(vector_resp, f"Vector/kNN ({vector_ms}ms)")
print_results(hybrid_resp, f"Hybrid RRF ({hybrid_ms}ms)")


  BM25 (82ms)  |  7 total  |  46ms
  [0.3688] arxiv:2301.00001 | section:Conclusion
           Title: Attention Is All You Need Abstract: We propose a novel architecture based on attention mechanisms...  Section: Co...
  [0.3629] arxiv:2301.00001 | section:Methods (part 3)
           Title: Attention Is All You Need Abstract: We propose a novel architecture based on attention mechanisms...  Section: Me...
  [0.3461] arxiv:2301.00001 | section:Abstract + Introduction
           Title: Attention Is All You Need Abstract: We propose a novel architecture based on attention mechanisms...  Section: Ab...
  [0.3410] arxiv:2301.00001 | section:Related Work
           Title: Attention Is All You Need Abstract: We propose a novel architecture based on attention mechanisms...  Section: Re...
  [0.2967] arxiv:2301.00001 | section:Results
           Title: Attention Is All You Need Abstract: We propose a novel architecture based on attention mechanisms...  Section: Re...

  Vector/kNN (81ms)  |  5

In [36]:
# Performance benchmark: compare latency across search modes
N_RUNS = 5

bm25_times, vector_times, hybrid_times = [], [], []

for _ in range(N_RUNS):
    t = time.time(); search_bm25_chunks(QUERY);   bm25_times.append((time.time()-t)*1000)
    t = time.time(); search_vector_chunks(query_embedding); vector_times.append((time.time()-t)*1000)
    t = time.time(); search_hybrid_chunks(QUERY, query_embedding); hybrid_times.append((time.time()-t)*1000)

print("\nLatency benchmark (5 runs each):")
print(f"  BM25:    {sum(bm25_times)/N_RUNS:.1f}ms avg")
print(f"  Vector:  {sum(vector_times)/N_RUNS:.1f}ms avg")
print(f"  Hybrid:  {sum(hybrid_times)/N_RUNS:.1f}ms avg")
print()
print("Note: embedding generation (~200ms) is not included above.")
print("At query time, the total hybrid latency is: embed_query + search ≈ 400ms")
print("BM25 has no embedding cost — use it when speed is critical.")


Latency benchmark (5 runs each):
  BM25:    8.3ms avg
  Vector:  10.4ms avg
  Hybrid:  13.3ms avg

Note: embedding generation (~200ms) is not included above.
At query time, the total hybrid latency is: embed_query + search ≈ 400ms
BM25 has no embedding cost — use it when speed is critical.


---

## Section 6: Graceful Degradation

### Concept: Hybrid → BM25 fallback

What happens if the Jina API is down? Or the JINA_API_KEY isn't set?

**Never let the embedding service failure break search.** BM25 already works — hybrid is an upgrade. The fallback ensures users still get results.

In [37]:
def smart_search(
    query_text: str,
    use_hybrid: bool = True,
    categories: list[str] | None = None,
    size: int = 10,
) -> dict:
    """
    Search that gracefully degrades from hybrid to BM25.

    Returns a dict with 'results' and 'search_mode' indicating which path was taken.
    The caller (API layer) can log search_mode for monitoring.
    """
    if use_hybrid:
        try:
            # Attempt to embed the query
            query_embedding = embed_texts([query_text], task="retrieval.query")[0]
            # Attempt hybrid search
            response = search_hybrid_chunks(query_text, query_embedding, size=size)
            return {"results": response, "search_mode": "hybrid"}
        except Exception as e:
            print(f"⚠️  Hybrid failed ({e}), falling back to BM25")
            # Fall through to BM25

    # BM25 fallback (or when use_hybrid=False)
    response = search_bm25_chunks(query_text, size=size)
    return {"results": response, "search_mode": "bm25"}


# Test fallback
result = smart_search("attention mechanism", use_hybrid=True)
print(f"Search mode used: {result['search_mode']}")
print(f"Results: {result['results']['hits']['total']['value']} hits")

# Force BM25 path
result_bm25 = smart_search("attention mechanism", use_hybrid=False)
print(f"\nForced BM25 mode: {result_bm25['search_mode']}")

Search mode used: hybrid
Results: 7 hits

Forced BM25 mode: bm25


### 💡 Key insight: `search_mode` in the API response

The API response includes `search_mode: "hybrid"` or `search_mode: "bm25"`. This tells the client:
- Which path was actually used (for monitoring)
- Whether to expect semantic matches or just keyword matches

You can track `search_mode` in your analytics to see how often the fallback is triggered — a high fallback rate signals the embedding service needs attention.

---

## Summary — What You Built

```
✅ Section-aware chunker   — 4 cases: merge / keep / split / paragraph-fallback
✅ Context headers         — title + abstract prepended to every chunk
✅ Jina v3 embeddings      — asymmetric retrieval.query vs retrieval.passage tasks
✅ Batched embedding        — 32 texts per API call, rate-limit safe
✅ chunks index             — knn_vector field with HNSW, 1024 dimensions
✅ RRF math                 — rank-based fusion, k=60, immune to score scale mismatch
✅ OpenSearch hybrid query  — native BM25 + kNN with normalization pipeline
✅ Graceful degradation     — hybrid → BM25 if embedding unavailable
✅ search_mode in response  — observable, monitorable
```

### Concepts mastered:
- **Why sections, not tokens**: chunking respects document structure
- **Context header impact**: single biggest quality improvement over naive chunking
- **Asymmetric embeddings**: query task ≠ passage task — both matter
- **HNSW parameters**: m and ef_construction tradeoffs
- **RRF vs score averaging**: ranks are stable, raw scores are not
- **k=60**: why it prevents top-rank domination
- **Separate chunks index**: flat structure for simple kNN, not nested vectors
- **Normalization pipeline**: required for native hybrid in OpenSearch
